# Financial Forecasting Frontier: Distributed ML

**Colab companion notebook** · PySpark analysis of the repository's `bank.csv`

[Open this notebook in Colab](https://colab.research.google.com/github/harshpaul1999-tech/banking-distributed-ml/blob/main/notebooks/Financial_Forecasting_Frontier_Distributed_ML.ipynb) · [View on GitHub](https://github.com/harshpaul1999-tech/banking-distributed-ml/blob/main/notebooks/Financial_Forecasting_Frontier_Distributed_ML.ipynb)

This is a compact, runnable adaptation of the [reference notebook](https://github.com/amol004/Financial-Forecasting-Frontier-Distributed-ML/blob/main/Financial_Forecasting_Frontier_Distributed_ML.ipynb). It loads the bank dataset from this repository instead of a personal Google Drive path, and retains the reference workflow: Spark EDA, Spark SQL, charts, hypothesis checks, Random Forest classification, event-time windows, and an optional Gradio demo.

The companion [PDF runbook](https://github.com/amol004/Financial-Forecasting-Frontier-Distributed-ML/blob/main/Banking%20Data%20Analysis%20with%20Hadoop%2CMapreduce%2C%20Apache%20Hive%20and%20PySpark.pdf) documents GCP Dataproc, HDFS, mrjob, and Hive. Those cloud services need a configured GCP cluster; Colab by itself does not start HDFS or Hive. The [main project notebook](https://colab.research.google.com/github/harshpaul1999-tech/banking-distributed-ml/blob/main/notebooks/Banking_Distributed_Analytics_Colab.ipynb) contains the Hadoop/Hive commands and the fuller five-part project implementation.

**Interpretation note:** the Random Forest section follows the reference's inclusion of `duration`. Because duration is known during or after the call, this model is a retrospective demonstration and should not be used as a pre-contact prediction model. The main notebook excludes it for that reason.


## 1. Colab setup and dataset

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/harshpaul1999-tech/banking-distributed-ml.git"
REPO_DIR = Path.cwd()
if not (REPO_DIR / "data" / "bank.csv").exists():
    REPO_DIR = Path("/content/banking-distributed-ml")
    if not (REPO_DIR / "data" / "bank.csv").exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)

# PySpark 3.5 works with Java 17. Install Java only if the runtime lacks it.
if shutil.which("java") is None and shutil.which("apt-get") is not None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "openjdk-17-jre-headless"], check=True)
if shutil.which("java") is None:
    raise RuntimeError("A Java runtime is required for PySpark.")
if not os.environ.get("JAVA_HOME"):
    java_path = Path(shutil.which("java")).resolve()
    for candidate in (java_path.parents[1], Path("/usr/lib/jvm/java-17-openjdk-amd64")):
        if candidate.exists():
            os.environ["JAVA_HOME"] = str(candidate)
            break

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.6"], check=True)
import pyspark
print("PySpark:", pyspark.__version__)

from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder
         .appName("FinancialForecastingFrontier")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "4")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
DATA_PATH = REPO_DIR / "data" / "bank.csv"
df = spark.read.csv(str(DATA_PATH), header=True, inferSchema=True)
print("Rows:", df.count(), "Columns:", len(df.columns))
df.show(5, truncate=False)


The source file is the 4,521-row, 17-column bank-marketing sample. The `y` field is the term-deposit subscription outcome. In this compact notebook, Spark reads the CSV directly and performs the analysis in local mode on the Colab runtime.


## 2. Schema, data quality, and descriptive profile

In [ ]:
df.printSchema()
print("Exact duplicate rows:", df.count() - df.distinct().count())
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)
df.describe("age", "balance", "duration", "campaign", "pdays", "previous").show()
df.groupBy("y").count().orderBy("y").show()


The `pdays = -1` value means the client was not previously contacted. It is a meaningful category in the source data, not an ordinary negative duration. The target is imbalanced, so classifier accuracy needs to be read alongside class-sensitive measures.


## 3. Spark aggregations and window operations

In [ ]:
job_summary = (df.groupBy("job")
    .agg(F.count("*").alias("customers"),
         F.round(F.avg("balance"), 2).alias("average_balance"),
         F.sum(F.when(F.col("y") == "yes", 1).otherwise(0)).alias("subscribers"))
    .withColumn("subscription_rate_pct",
                F.round(100 * F.col("subscribers") / F.col("customers"), 2))
    .orderBy(F.desc("subscription_rate_pct")))
job_summary.show(truncate=False)

from pyspark.sql.window import Window
job_age_window = Window.partitionBy("job").orderBy("age").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df.select("job", "age", "balance",
          F.sum("balance").over(job_age_window).alias("running_balance_by_job"))\
  .show(12, truncate=False)


The aggregation creates a small summary table by job. The window expression partitions rows by job, orders each partition by age, and builds a cumulative balance. Spark can distribute the grouped work across partitions; this local demonstration does not measure cluster speed-up.


## 4. Spark SQL: customer segments

In [ ]:
df.createOrReplaceTempView("bank_table")

# High-balance customer profile by marital status
spark.sql("""
SELECT marital, COUNT(*) AS customers,
       ROUND(AVG(balance), 2) AS average_balance,
       ROUND(AVG(duration), 2) AS average_call_seconds
FROM bank_table
WHERE balance > 2000
GROUP BY marital
ORDER BY average_balance DESC
""").show()

# Term-deposit conversion rate by education
spark.sql("""
SELECT education, COUNT(*) AS contacts,
       SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) AS subscriptions,
       ROUND(100.0 * SUM(CASE WHEN y = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_pct
FROM bank_table
GROUP BY education
ORDER BY conversion_pct DESC
""").show()

# Age bands among subscribers
spark.sql("""
SELECT CASE WHEN age < 30 THEN 'Under 30'
            WHEN age <= 50 THEN '30-50' ELSE 'Over 50' END AS age_band,
       COUNT(*) AS subscribers, ROUND(AVG(balance), 2) AS average_balance
FROM bank_table WHERE y = 'yes'
GROUP BY CASE WHEN age < 30 THEN 'Under 30'
              WHEN age <= 50 THEN '30-50' ELSE 'Over 50' END
ORDER BY subscribers DESC
""").show()


These queries summarize this campaign sample. A higher observed rate is not automatically evidence that a segment will respond the same way in a future campaign; check sample sizes and validate against newer data before using a result operationally.


## 5. Representative visualizations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Aggregate in Spark first; only the small plot tables are collected to Pandas.
age_bins = (df.withColumn("age_band", (F.floor(F.col("age") / 10) * 10).cast("int"))
    .groupBy("age_band", "y").count().toPandas())
wide = age_bins.pivot(index="age_band", columns="y", values="count").fillna(0)
wide.plot(kind="bar", figsize=(10, 5), color={"no":"#7aa6c2", "yes":"#e58b55"})
plt.title("Bank contacts and term-deposit outcomes by age decade")
plt.xlabel("Age decade"); plt.ylabel("Contacts"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

job_plot = job_summary.select("job", "subscription_rate_pct", "customers").toPandas().sort_values("subscription_rate_pct")
plt.figure(figsize=(9, 6))
plt.barh(job_plot["job"], job_plot["subscription_rate_pct"], color="#4c8c78")
plt.title("Observed subscription rate by job")
plt.xlabel("Subscription rate (%)"); plt.tight_layout(); plt.show()

monthly = (df.groupBy("month")
    .agg(F.count("*").alias("contacts"), F.avg(F.when(F.col("y") == "yes", 1).otherwise(0)).alias("rate"))
    .orderBy(F.desc("contacts")).toPandas())
monthly["rate_pct"] = monthly["rate"] * 100
monthly


The figures are descriptive. The source notebook presents ten different chart experiments; this companion keeps three representative views to make the runnable walkthrough shorter. When recording, point to the denominators as well as the percentages, especially for small job or month groups.


## 6. Simple hypothesis checks

In [ ]:
# Two-group mean comparison for call duration (Welch's t-test)
# Chi-square tests for age-band and previous-outcome association with y.
from scipy.stats import ttest_ind, chi2_contingency

local = df.select("age", "duration", "poutcome", "y").toPandas()
yes_duration = local.loc[local.y == "yes", "duration"]
no_duration = local.loc[local.y == "no", "duration"]
t_stat, t_p = ttest_ind(yes_duration, no_duration, equal_var=False)
print(f"Duration Welch t-test: t={t_stat:.3f}, p={t_p:.3g}")

local["age_band"] = pd.cut(local.age, bins=[0, 29, 50, 120], labels=["Under 30", "30-50", "Over 50"])
age_table = pd.crosstab(local["age_band"], local["y"])
age_chi2, age_p, _, _ = chi2_contingency(age_table)
print(f"Age band vs subscription chi-square: chi2={age_chi2:.3f}, p={age_p:.3g}")

known_outcomes = local[local.poutcome.isin(["success", "failure"])]
outcome_table = pd.crosstab(known_outcomes["poutcome"], known_outcomes["y"])
out_chi2, out_p, _, _ = chi2_contingency(outcome_table)
print(f"Prior outcome vs subscription chi-square: chi2={out_chi2:.3f}, p={out_p:.3g}")


These tests check association in one dataset. They do not prove that a variable causes a subscription, and statistical significance does not measure practical value. The reference notebook uses one-way ANOVA for some binary outcomes; here, chi-square tests are used for categorical-by-categorical comparisons, and Welch's t-test compares the two duration groups.


## 7. Spark ML Random Forest

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# Encode the positive class explicitly so probability index 1 means subscription.
model_df = df.withColumn("label", F.when(F.col("y") == "yes", 1.0).otherwise(0.0))
cat_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols],
                        outputCols=[f"{c}_oh" for c in cat_cols], handleInvalid="keep")
numeric_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
assembler = VectorAssembler(inputCols=numeric_cols + [f"{c}_oh" for c in cat_cols],
                            outputCol="features", handleInvalid="keep")
rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                             numTrees=100, seed=42)
pipeline = Pipeline(stages=indexers + [encoder, assembler, rf])
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
rf_model = pipeline.fit(train_df)
predictions = rf_model.transform(test_df)
roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(predictions)
pr = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR").evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(predictions)
print(f"Test ROC-AUC: {roc:.4f}")
print(f"Test PR-AUC: {pr:.4f}")
print(f"Test accuracy: {accuracy:.4f}")
print(f"All-no baseline accuracy: {1.0 - model_df.select(F.avg('label')).first()[0]:.4f}")
predictions.select("y", "prediction", "probability").show(12, truncate=False)


This reproduces the reference's Random Forest approach with the target encoded explicitly (`no = 0`, `yes = 1`) and reports ROC-AUC, PR-AUC, and accuracy beside an all-no baseline. The held-out result is specific to the random split. `duration` is retained to match the reference workflow, but it is only known during/after the call; for a pre-contact model, use the Logistic Regression pipeline in the main notebook, which excludes it.


## 8. Streaming windows and per-record predictions

In [ ]:
# The repository includes four small CSV chunks with synthetic event_time values.
# They demonstrate event-time windows; timestamps are not present in the source bank.csv.
from pyspark.sql import types as T
from pyspark.sql.functions import window
import tempfile

stream_schema = df.schema.add(T.StructField("event_id", T.StringType(), True))\
                         .add(T.StructField("event_time", T.TimestampType(), True))
stream_dir = REPO_DIR / "data" / "stream_sample"
base_stream = (spark.readStream.option("header", True).schema(stream_schema)
               .option("maxFilesPerTrigger", 1).csv(str(stream_dir)))
stream_events = base_stream.withWatermark("event_time", "1 minute")
windowed = (stream_events.groupBy(window("event_time", "10 seconds"), "job")
            .agg(F.count("*").alias("transactions"),
                 F.round(F.avg("balance"), 2).alias("average_balance")))
checkpoint_root = Path(tempfile.mkdtemp(prefix="frontier_stream_"))
window_query = (windowed.writeStream.outputMode("update").format("console")
                .option("truncate", False)
                .option("checkpointLocation", str(checkpoint_root / "windows"))
                .trigger(availableNow=True).start())
window_query.awaitTermination()


In [ ]:
# Score each incoming micro-batch with the fitted Random Forest.
def show_predictions(batch, batch_id):
    print(f"Prediction micro-batch {batch_id}")
    batch.select("event_id", "event_time", "job", "balance", "prediction", "probability")\
         .show(10, truncate=False)

prediction_stream = rf_model.transform(base_stream)
prediction_query = (prediction_stream.writeStream.foreachBatch(show_predictions)
                    .option("checkpointLocation", str(checkpoint_root / "predictions"))
                    .trigger(availableNow=True).start())
prediction_query.awaitTermination()


The sample stream is a finite replay, not live banking traffic. Event times are synthetic and intentionally include out-of-order records. The one-minute watermark bounds how long Spark retains window state for late data; records later than the bound are not guaranteed to revise old windows.


## 9. Optional Gradio prediction form

In [ ]:
# Optional Colab UI. Run this cell only if you want to interact with a sample prediction.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
import gradio as gr

def predict_subscription(row):
    one_row = spark.createDataFrame([row])
    result = rf_model.transform(one_row).select("prediction", "probability").first()
    label = "yes" if float(result["prediction"]) == 1.0 else "no"
    confidence = float(result["probability"][int(result["prediction"])])
    return {"predicted_subscription": label, "class_confidence": round(confidence, 4)}

sample_customer = {"age": 42, "job": "management", "marital": "married",
    "education": "tertiary", "default": "no", "balance": 1500, "housing": "yes",
    "loan": "no", "contact": "cellular", "day": 15, "month": "may",
    "duration": 180, "campaign": 1, "pdays": -1, "previous": 0, "poutcome": "unknown"}
demo = gr.Interface(fn=predict_subscription,
                     inputs=gr.JSON(value=sample_customer, label="Customer fields"),
                     outputs=gr.JSON(label="Model response"),
                     title="Term Deposit Subscription Demo")
demo.launch(share=False)


The Gradio form is an educational demo that runs in the current notebook session. It does not authenticate users, persist customer data, or serve as a production banking decision system. Review the main project README for the complete assignment notebook, HDFS/Hive instructions, and the stronger streaming implementation.


## Reference links and closing notes

- [Reference PDF: Banking Data Analysis with Hadoop, MapReduce, Apache Hive and PySpark](https://github.com/amol004/Financial-Forecasting-Frontier-Distributed-ML/blob/main/Banking%20Data%20Analysis%20with%20Hadoop%2CMapreduce%2C%20Apache%20Hive%20and%20PySpark.pdf)
- [Reference Financial Forecasting Frontier notebook](https://github.com/amol004/Financial-Forecasting-Frontier-Distributed-ML/blob/main/Financial_Forecasting_Frontier_Distributed_ML.ipynb)
- [Main five-part project notebook](https://colab.research.google.com/github/harshpaul1999-tech/banking-distributed-ml/blob/main/notebooks/Banking_Distributed_Analytics_Colab.ipynb)
- [Video explanation script](https://github.com/harshpaul1999-tech/banking-distributed-ml/blob/main/docs/reference_colab_video_script.md)

The reference PDF explains cluster operations; this Colab companion covers the Spark notebook workflow. Use the main project notebook for the complete five-part deliverable and for its documented local-vs-cluster boundaries.
